# 03 — Advanced Vector Search

**Goal**: Understand approximate nearest neighbor (ANN) search, indexing strategies, and production considerations.

In [ ]:
import chromadb
import requests
import numpy as np
import time

client = chromadb.PersistentClient(path="./chroma_data")

def get_embedding(text):
    resp = requests.post("http://localhost:11434/api/embeddings", json={
        "model": "nomic-embed-text",
        "prompt": text
    })
    return resp.json()["embedding"]

print("Ready")

## 1. Brute Force vs. ANN (Approximate Nearest Neighbor)

**Brute Force**: Compare query with EVERY document (100% accurate, O(n))  
**ANN**: Use an index to find approximate neighbors (99% accurate, O(log n))

| Method | Accuracy | Speed | Memory |
|---|---|---|---|
| Brute Force | 100% | Slow for millions | Low |
| HNSW (ANN) | ~99% | Very fast | High |
| IVF (ANN) | ~98% | Fast | Medium |

ChromaDB uses **HNSW** (Hierarchical Navigable Small World) by default.

## 2. Creating a Large Test Dataset

Let's create 1000 synthetic documents to demonstrate scale.

In [ ]:
# Generate 1000 documents about various topics
topics = ["machine learning", "deep learning", "NLP", "computer vision", "robotics"]
adjectives = ["advanced", "basic", "modern", "classic", "emerging"]
nouns = ["techniques", "algorithms", "methods", "approaches", "frameworks"]

large_collection = client.get_or_create_collection(
    name="large_test",
    metadata={"hnsw:space": "cosine"}  # explicitly use cosine distance
)

import random
random.seed(42)

batch_size = 100
total_docs = 1000

for batch_start in range(0, total_docs, batch_size):
    batch_end = min(batch_start + batch_size, total_docs)
    ids, texts, embeddings, metadatas = [], [], [], []
    
    for i in range(batch_start, batch_end):
        topic = random.choice(topics)
        adj = random.choice(adjectives)
        noun = random.choice(nouns)
        text = f"{adj} {topic} {noun}: document {i}"
        
        ids.append(f"doc_{i}")
        texts.append(text)
        embeddings.append(get_embedding(text))
        metadatas.append({"topic": topic, "index": i})
    
    large_collection.add(ids=ids, embeddings=embeddings,
                        documents=texts, metadatas=metadatas)

print(f"Added {total_docs} documents to 'large_test' collection")

## 3. Query Performance

Let's measure search speed.

In [ ]:
queries = [
    "advanced machine learning techniques",
    "basic NLP algorithms",
    "modern computer vision methods"
]

for query in queries:
    query_emb = get_embedding(query)
    
    start = time.time()
    results = large_collection.query(
        query_embeddings=[query_emb],
        n_results=5
    )
    elapsed = time.time() - start
    
    print(f"\nQuery: {query}")
    print(f"Time: {elapsed*1000:.1f}ms")
    for doc in results['documents'][0][:3]:
        print(f"  → {doc}")

## 4. Distance Metrics in ChromaDB

ChromaDB supports three distance metrics:

In [ ]:
# Create collections with different distance metrics
for metric in ["cosine", "l2", "ip"]:
    col = client.get_or_create_collection(
        name=f"test_{metric}",
        metadata={"hnsw:space": metric}
    )
    
    # Add same documents
    emb = get_embedding("test document")
    col.add(ids=["test"], embeddings=[emb], documents=["test document"])
    print(f"Created collection with {metric} distance")

print("\nDistance metrics available:")
print("  cosine: 1 - cos(a,b) [default, recommended]")
print("  l2:     Euclidean distance")
print("  ip:     Inner product (dot product)")

## 5. HNSW Parameters

HNSW has two key parameters that trade off speed vs. accuracy:

In [ ]:
# Create collection with custom HNSW parameters
optimized_collection = client.get_or_create_collection(
    name="optimized_hnsw",
    metadata={
        "hnsw:space": "cosine",
        "hnsw:construction_ef": 200,   # higher = better quality index (slower build)
        "hnsw:M": 32,                   # higher = better recall (more memory)
        "hnsw:search_ef": 100,          # higher = better search quality (slower)
    }
)

print("Collection with tuned HNSW parameters:")
print(f"  construction_ef: 200 (build quality)")
print(f"  M: 32 (connections per node)")
print(f"  search_ef: 100 (search depth)")
print()
print("Default values: construction_ef=100, M=16, search_ef=10")

## 6. Production Considerations

### When to use what:

| Use Case | Solution |
|---|---|
| <10K docs, prototyping | ChromaDB with defaults |
| 10K-1M docs | ChromaDB with tuned HNSW |
| 1M+ docs | Heavy-duty: Qdrant, Weaviate, Pinecone |
| High availability | Cloud vector DB with replication |
| Hybrid search (keyword + semantic) | PostgreSQL + pgvector |

### Best practices:
1. **Normalize embeddings** before storing
2. **Use cosine distance** for text embeddings
3. **Batch inserts** (100-1000 at a time)
4. **Store metadata** for pre-filtering
5. **Monitor recall** (you want >95% ANN accuracy)

## Key Takeaways

1. **ANN** = approximate search (fast, ~99% accurate)
2. **HNSW** = default algorithm in ChromaDB (excellent all-rounder)
3. **Distance metrics**: cosine for text, L2 for images, IP for normalized
4. **Tune HNSW params** for your speed/accuracy needs
5. Vector DBs scale to millions with proper indexing

**Next**: LlamaIndex — connect vector search to LLMs →